# Z0：不要只相信训练 loss

这一份 Notebook 不训练新架构。我们用一个可完全计算的小世界练习多步、反事实、OOD、不确定性和运行证据。之后把同一套问题移到自己的 PA1。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'hwm').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
from hwm.evaluation import horizon_errors, counterfactual_sensitivity, calibration_bins, RunManifest, RunTimer, runtime_summary


## 1. 一个故意有偏差的模型

真实世界每次动作完整移动；模型只预测 0.9 倍位移。一步误差很小，多步以后会逐渐落后。

In [ ]:
def rollout(start, actions, scale=1.0):
    result, state = [], float(start)
    for action in actions:
        state += scale * action; result.append(state)
    return np.asarray(result)
starts = [0.0, 1.0, -1.0]
action_sequences = [[1.0] * 12, [-1.0] * 12, [1.0, -1.0] * 6]
truth = np.stack([rollout(s, a, 1.0) for s, a in zip(starts, action_sequences)])
errors = horizon_errors(lambda s, a: rollout(s, a, 0.9), starts, action_sequences, truth)
print('horizon 1/5/12 MSE:', [round(float(errors[i]), 4) for i in [0, 4, 11]])
assert errors[-1] > errors[0]


## 2. 固定起点，只替换动作

如果所有动作得到同一未来，模型可能只在复制惯性。反事实一次只改动作，避免把初始状态差异混进来。

In [ ]:
candidate_actions = [[0, 0, 0], [1, 1, 1], [-1, -1, -1]]
sensitivity = counterfactual_sensitivity(lambda s, a: rollout(s, a, 0.9), 0.0, candidate_actions)
print('相对 stay 的未来差异:', np.round(sensitivity, 3))
assert sensitivity[1] > 0 and sensitivity[2] > 0


## 3. OOD：训练范围之外会怎样

模型只在动作绝对值不超过 1 的数据上拟合。现在测试动作 3。模型仍给出数字，不代表这个数字可靠。

In [ ]:
in_distribution = abs(rollout(0, [1], 0.9)[0] - rollout(0, [1], 1.0)[0])
ood = abs(rollout(0, [3], 0.7)[0] - rollout(0, [3], 1.0)[0])
print('ID/OOD absolute error:', round(float(in_distribution), 3), round(float(ood), 3))
assert ood > in_distribution


## 4. 不确定性是否说真话

如果模型说 80% 会碰撞，很多这样的样本中大约应有 80% 真碰撞。分箱能看出置信度与实际频率是否一致。

In [ ]:
probabilities = np.array([0.1, 0.2, 0.35, 0.65, 0.8, 0.95])
outcomes = np.array([0, 0, 1, 0, 1, 1])
bins = calibration_bins(probabilities, outcomes, num_bins=3)
for item in bins: print(item)
assert len(bins) == 3


## 5. Planner 会主动寻找模型漏洞

若模型错误地认为超大动作没有代价，Planner 可能选择训练数据从未出现的动作。评价时要限制动作范围，并在真实环境复核选中的候选。

In [ ]:
candidates = np.linspace(-4, 4, 33)
model_scores = -(0.9 * candidates - 3.0) ** 2
chosen = float(candidates[model_scores.argmax()])
print('planner chosen action:', chosen, 'training support: [-1,1]')
print('是否 OOD:', abs(chosen) > 1)
assert abs(chosen) > 1


## 6. 一次运行应留下什么

状态页不能只写‘在 24GB 上能跑’。运行清单要保存数据、seed、命令、显卡、时间、峰值显存和 checkpoint 哈希。

In [ ]:
with RunTimer() as timer:
    _ = sum(range(1000))
manifest = RunManifest(experiment='Z0-smoke', route='Z', seed=0, dataset='analytic-toy', split='test', command='run Z0 notebook', started_at='2026-08-12T00:00:00Z', wall_time_seconds=timer.seconds, notes='CPU 教学 smoke，不是 24GB 证据')
print(runtime_summary())
print(manifest)
assert manifest.peak_reserved_mb is None


## 小结

多步曲线检查误差累积，反事实检查动作作用，OOD 检查数据边界，校准检查模型是否知道自己不知道，真实闭环检查模型是否真的帮助行动。PA2 从其中一个稳定失败开始。